In [1]:
!lsb_release -a

No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.4 LTS
Release:	22.04
Codename:	jammy


In [2]:
!python --version

Python 3.12.12


In [3]:
!sudo apt-get update && sudo apt-get install -y gnupg software-properties-common
!sudo apt-key adv --fetch-key https://repo.download.nvidia.com/jetson/jetson-ota-public.asc
!sudo add-apt-repository -y 'deb https://repo.download.nvidia.com/jetson/x86_64/noble r39.2 main'


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [119 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Err:2 https://cli.github.com/packages stable InRelease
  The following signatures couldn't be verified because the public key is not available: NO_PUBKEY 5612B36462313325
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [3,227 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://ppa.launchpadconten

In [4]:
!sudo apt-get update
!sudo apt-get install -y libnvvpi4 python3.12 python3.12-vpi4 python3.12-venv

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://repo.download.nvidia.com/jetson/x86_64/noble r39.2 InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Err:2 https://cli.github.com/packages stable InRelease
  The following signatures couldn't be verified because the public key is not available: NO_PUBKEY 5612B36462313325
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa

In [6]:
!pip install -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.8/103.8 kB 6.5 MB/s eta 0:00:00


In [5]:
import vpi
print(vpi.__version__)

4.1.4


In [7]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.215 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 1.3/112.6 GB disk)


In [9]:
import torch
import torch.nn as nn

path = "/content/team_air_gcp_localization_protoype.pth"

class ConvNet(nn.Module):
  def __init__(self):
        super().__init__()
        # Input 94 x 80 x 1
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=0),  # 92 x 78 x 8
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)       # 46 x 39 x 8
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(8, 8, kernel_size=3, groups=8, padding=0), # 44 x 37 x 8
            nn.Conv2d(8, 12, kernel_size=1, bias=False), # Depthwise separable convolution
            nn.BatchNorm2d(12),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)               # 22 x 18 x 12
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(12, 12, kernel_size=3, groups=12, padding=0), # 20 x 16 x 12
            nn.Conv2d(12, 16, kernel_size=1, bias=False), # Depthwise separable convolution
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)                   # 10 x 8 x 16
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=3, groups=16, padding=0), # 8 x 6 x 16
            nn.Conv2d(16, 24, kernel_size=1),
            nn.BatchNorm2d(24),
            nn.ReLU()                                               # 8 x 6 x 24
        )

        # gap might be a bad idea
        # but maybe a smaller adaptive pooling layer will work better
        self.pool = nn.AdaptiveAvgPool2d((2, 2))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(24 * 2 * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 10),
        )

  def forward(self, x):
        # Input in (Batch, Channels, Height, Width)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print(device)

cnn = ConvNet()
cnn.load_state_dict(torch.load(path, weights_only=True, map_location=device))
cnn = cnn.to(device)
cnn.eval()

cuda


ConvNet(
  (conv1): Sequential(
    (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1), groups=8)
    (1): Conv2d(8, 12, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (2): BatchNorm2d(12, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv3): Sequential(
    (0): Conv2d(12, 12, kernel_size=(3, 3), stride=(1, 1), groups=12)
    (1): Conv2d(12, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv4)

In [11]:
from pathlib import Path
import math
import cv2 as cv
import numpy as np
import torch
import torchvision.transforms.functional as TF
from ultralytics import YOLO
import vpi

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

model = YOLO("/content/best.pt")

"""Load template matching files on CPU"""
directory = Path("/content/Numbers")
files = sorted(
    [f for f in directory.iterdir() if f.is_file() and f.suffix.lower() == ".jpg"]
)
templates = [cv.imread(str(entry))[:, :, 1] for entry in files]

# Exact 3x3 affine matrices for 90-degree rotations of a 256x128 (W x H) image -> 128x256
ROT90_CW_XFORM = np.array(
    [[0.0, -1.0, 127.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32
)
ROT90_CCW_XFORM = np.array(
    [[0.0, 1.0, 0.0], [-1.0, 0.0, 255.0], [0.0, 0.0, 1.0]], dtype=np.float32
)


"""Find Squares Algo"""
def extract_corners(polygon: np.ndarray, shape: tuple[int, ...]) -> np.ndarray | None:
    h, w = shape[:2]
    x = polygon[:, 0]
    y = polygon[:, 1]

    threshold = 0.01
    valid = (
        threshold * w <= x.min()
        and x.max() < (1 - threshold) * w
        and threshold * h <= y.min()
        and y.max() < (1 - threshold) * h
    )

    if not valid:
        print("Captured image to close to borders, skipping")
        return None

    contour = np.asarray(polygon, dtype=np.float32).reshape(-1, 1, 2)
    perimeter = cv.arcLength(contour, True)

    for epsilon_factor in (0.01, 0.02, 0.03, 0.05):
        approx = cv.approxPolyDP(contour, epsilon_factor * perimeter, True)
        if len(approx) == 4:
            if epsilon_factor == 0.05:
                print("Received high noise")
            return approx.reshape(4, 2)

    print("Captured non-square; likely partially captured")
    return None


def rectify_target(vpi_image: vpi.Image, corners: np.ndarray) -> vpi.Image:
    # Sort corners cyclically around their centroid
    center = corners.mean(axis=0)
    angles = np.arctan2(corners[:, 1] - center[1], corners[:, 0] - center[0])
    corners = corners[np.argsort(angles)]

    destination = np.array(
        [[0, 0], [255, 0], [255, 255], [0, 255]],
        dtype=np.float32,
    )

    # 3x3 homography computed on CPU; perspective warp executed on VPI CUDA backend
    H = cv.getPerspectiveTransform(corners.astype(np.float32), destination)
    with vpi.Backend.CUDA:
        return vpi_image.perspwarp(
            H,
            size=(256, 256),
            interp=vpi.Interp.LINEAR,
            border=vpi.Border.ZERO,
        )


"""Display Numbers from Square Algo"""
def analyze_squares(vpi_rectified: vpi.Image, padding: int = 10) -> tuple[str, str]:
    N = 256
    size_sq = 48
    center = (N - size_sq) // 2

    # rlock_cpu() exposes a zero-copy unified-memory read view on Jetson Orin Nano
    with vpi_rectified.rlock_cpu() as image:
        regions = {
            "top": image[padding : padding + size_sq, center : center + size_sq],
            "bottom": image[
                N - padding - size_sq : N - padding, center : center + size_sq
            ],
            "left": image[center : center + size_sq, padding : padding + size_sq],
            "right": image[
                center : center + size_sq, N - padding - size_sq : N - padding
            ],
        }
        means = {name: float(region.mean()) for name, region in regions.items()}

    horizontal = means["left"] + means["right"]
    vertical = means["top"] + means["bottom"]

    if horizontal > vertical:
        axis = "horizontal"
        candidates = ("left", "right")
    else:
        axis = "vertical"
        candidates = ("top", "bottom")

    number_side = min(candidates, key=lambda x: means[x])
    return axis, number_side


def retrieve_numbers(vpi_rectified: vpi.Image) -> np.ndarray:
    axis, number_side = analyze_squares(vpi_rectified)

    with vpi.Backend.CUDA:
        if axis == "horizontal":
            # Zero-copy 128x256 ROI view via vpi.RectangleI(x, y, width, height)
            rect = (
                vpi.RectangleI(128, 0, 128, 256)
                if number_side == "right"
                else vpi.RectangleI(0, 0, 128, 256)
            )
            vpi_half = vpi_rectified.view(rect)
            if number_side == "left":
                vpi_half = vpi_half.image_flip(vpi.Flip.BOTH)
        else:
            # Zero-copy 256x128 ROI view via vpi.RectangleI(x, y, width, height)
            rect = (
                vpi.RectangleI(0, 128, 256, 128)
                if number_side == "bottom"
                else vpi.RectangleI(0, 0, 256, 128)
            )
            vpi_half = vpi_rectified.view(rect)
            xform = ROT90_CW_XFORM if number_side == "top" else ROT90_CCW_XFORM
            vpi_half = vpi_half.perspwarp(
                xform,
                size=(128, 256),
                interp=vpi.Interp.NEAREST,
                border=vpi.Border.ZERO,
            )

        # Zero-copy final crop: x in [40, 120) -> w=80, y in [74, 168) -> h=94
        vpi_num = vpi_half.view(vpi.RectangleI(40, 74, 80, 94))

    # Synchronize and copy the 94x80 patch to a contiguous CPU NumPy array
    return np.ascontiguousarray(vpi_num.cpu())


"""Classification with CPU template matching"""
def conf_func(diff: float, val: float) -> float:
    return ((2 / (1 + math.exp(-20 * diff))) - 1) * (
        (2 / (1 + math.exp(-4.7 * val))) - 1
    )


def template_match(img: np.ndarray) -> tuple[int, float]:
    scores = []
    for template in templates:
        res = cv.matchTemplate(img, template, cv.TM_CCOEFF_NORMED)
        scores.append(cv.minMaxLoc(res)[1])

    arr = sorted(scores)
    return int(np.argmax(scores)), conf_func(arr[-1] - arr[-2], arr[-1])


def return_points(results, image: np.ndarray) -> None:
    # Extract single-channel green plane if a 3-channel BGR image is passed
    green = image[:, :, 1] if image.ndim == 3 else image
    vpi_frame = vpi.asimage(np.ascontiguousarray(green), vpi.Format.U8)

    for i, result in enumerate(results):
        if result.masks is None:
            print(f"No results found in image {i}")
            continue

        for polygon in result.masks.xy:
            corners = extract_corners(polygon, green.shape)
            if corners is None:
                continue

            centroid = np.mean(corners, axis=0)  # (x, y) in pixels
            vpi_rectified = rectify_target(vpi_frame, corners)
            num_img = retrieve_numbers(vpi_rectified)
            num, conf = template_match(num_img)
            mod = "Tem"

            if conf < 0.85:
                num_tensor = TF.to_tensor(num_img).unsqueeze(0).to(device)
                with torch.inference_mode():
                    cnn_output = cnn(num_tensor)
                    probs = torch.softmax(cnn_output, dim=1)
                    cnn_conf, cnn_num = torch.max(probs, dim=1)
                    if cnn_conf.item() > conf:
                        mod = "CNN"
                        num = int(cnn_num.item())
                        conf = float(cnn_conf.item())

            if conf < 0.7:
                print("Cannot reliably detect image; skipping")
                continue

            print(f"Number: {num}\nPixel coordinates: {centroid}\nModel: {mod}\nConfidence: {conf:.4f}\n\n")

In [22]:
img = cv.imread("/content/test_hard.jpg")
results = model(img, conf=0.74)
return_points(results, img[:, :, 1])


0: 640x480 6 ps, 14.0ms
Speed: 3.8ms preprocess, 14.0ms inference, 2.2ms postprocess per image at shape (1, 3, 640, 480)
Number: 5
Pixel coordinates: [     2277.4        3035]
Model: Tem
Confidence: 0.9398


Number: 1
Pixel coordinates: [      648.9      2239.6]
Model: Tem
Confidence: 0.9757


Number: 2
Pixel coordinates: [      793.8      1352.9]
Model: Tem
Confidence: 0.9023


Number: 4
Pixel coordinates: [     332.32      1403.3]
Model: Tem
Confidence: 0.8744


Number: 6
Pixel coordinates: [     2236.5      2211.3]
Model: CNN
Confidence: 0.9990


Number: 3
Pixel coordinates: [     1883.7      1126.1]
Model: Tem
Confidence: 0.9224




In [21]:
img = cv.imread("/content/test_hard.jpg")
vpi_frame = vpi.asimage(np.ascontiguousarray(img), vpi.Format.RGB8)
vpi_image = vpi_frame.rescale((640, 640), interp=vpi.Interp.LINEAR, backend=vpi.Backend.CUDA)
with vpi_image.rlock_cuda() as cuda_buffer:
    # Zero-copy wrap into a PyTorch GPU tensor (Shape: H, W, C)
    tensor_hwc = torch.as_tensor(cuda_buffer, device="cuda")

    # Convert HWC -> BCHW and normalize to [0.0, 1.0] for YOLO
    tensor_bchw = tensor_hwc.permute(2, 0, 1).unsqueeze(0).float().div_(255.0)

    # Run YOLO inference directly on the GPU tensor
    results = model(tensor_bchw, conf=0.74)


0: 640x640 6 ps, 13.9ms
Speed: 0.0ms preprocess, 13.9ms inference, 2.7ms postprocess per image at shape (1, 3, 640, 640)


In [28]:
import time

def return_points(results, image: np.ndarray) -> None:
    # Extract single-channel green plane if a 3-channel BGR image is passed
    green = image[:, :, 1] if image.ndim == 3 else image
    vpi_frame = vpi.asimage(np.ascontiguousarray(green), vpi.Format.U8)

    for i, result in enumerate(results):
        if result.masks is None:
            print(f"No results found in image {i}")
            continue

        for polygon in result.masks.xy:
            corners = extract_corners(polygon, green.shape)
            if corners is None:
                continue

            start_time = time.perf_counter()
            centroid = np.mean(corners, axis=0)  # (x, y) in pixels
            vpi_rectified = rectify_target(vpi_frame, corners)
            num_img = retrieve_numbers(vpi_rectified)
            num, conf = template_match(num_img)
            print(f"Template matching time: {(time.perf_counter() - start_time)*1000:.4f} ms")
            mod = "Tem"

            if conf < 0.85:
                num_tensor = TF.to_tensor(num_img).unsqueeze(0).to(device)
                with torch.inference_mode():
                    cnn_output = cnn(num_tensor)
                    probs = torch.softmax(cnn_output, dim=1)
                    cnn_conf, cnn_num = torch.max(probs, dim=1)
                    if cnn_conf.item() > conf:
                        mod = "CNN"
                        num = int(cnn_num.item())
                        conf = float(cnn_conf.item())

            if conf < 0.7:
                print("Cannot reliably detect image; skipping")
                continue

            print(f"Number: {num}\nPixel coordinates: {centroid}\nModel: {mod}\nConfidence: {conf:.4f}\n\n")

In [26]:
img = cv.imread("/content/test_hard.jpg")
results = model(img, conf=0.74)
return_points(results, img[:, :, 1])


0: 640x480 6 ps, 17.9ms
Speed: 4.7ms preprocess, 17.9ms inference, 2.9ms postprocess per image at shape (1, 3, 640, 480)
Template matching time: 13.7768 ms
Number: 5
Pixel coordinates: [     2277.4        3035]
Model: Tem
Confidence: 0.9398


Template matching time: 11.0299 ms
Number: 1
Pixel coordinates: [      648.9      2239.6]
Model: Tem
Confidence: 0.9757


Template matching time: 8.7345 ms
Number: 2
Pixel coordinates: [      793.8      1352.9]
Model: Tem
Confidence: 0.9023


Template matching time: 11.2364 ms
Number: 4
Pixel coordinates: [     332.32      1403.3]
Model: Tem
Confidence: 0.8744


Template matching time: 9.9747 ms
Number: 6
Pixel coordinates: [     2236.5      2211.3]
Model: CNN
Confidence: 0.9990


Template matching time: 9.2991 ms
Number: 3
Pixel coordinates: [     1883.7      1126.1]
Model: Tem
Confidence: 0.9224




In [40]:
import time

def retrieve_numbers(vpi_rectified: vpi.Image) -> vpi.Image:
    axis, number_side = analyze_squares(vpi_rectified)

    with vpi.Backend.CUDA:
        if axis == "horizontal":
            # Zero-copy 128x256 ROI view via vpi.RectangleI(x, y, width, height)
            rect = (
                vpi.RectangleI(128, 0, 128, 256)
                if number_side == "right"
                else vpi.RectangleI(0, 0, 128, 256)
            )
            vpi_half = vpi_rectified.view(rect)
            if number_side == "left":
                vpi_half = vpi_half.image_flip(vpi.Flip.BOTH)
        else:
            # Zero-copy 256x128 ROI view via vpi.RectangleI(x, y, width, height)
            rect = (
                vpi.RectangleI(0, 128, 256, 128)
                if number_side == "bottom"
                else vpi.RectangleI(0, 0, 256, 128)
            )
            vpi_half = vpi_rectified.view(rect)
            xform = ROT90_CW_XFORM if number_side == "top" else ROT90_CCW_XFORM
            vpi_half = vpi_half.perspwarp(
                xform,
                size=(128, 256),
                interp=vpi.Interp.NEAREST,
                border=vpi.Border.ZERO,
            )

        # Zero-copy final crop: x in [40, 120) -> w=80, y in [74, 168) -> h=94
        vpi_num = vpi_half.view(vpi.RectangleI(40, 74, 80, 94))

    # Synchronize and copy the 94x80 patch to a contiguous CPU NumPy array
    return vpi_num

def return_points(results, image: np.ndarray) -> None:
    # Extract single-channel green plane if a 3-channel BGR image is passed
    green = image[:, :, 1] if image.ndim == 3 else image
    vpi_frame = vpi.asimage(np.ascontiguousarray(green), vpi.Format.U8)

    for i, result in enumerate(results):
        if result.masks is None:
            print(f"No results found in image {i}")
            continue

        for polygon in result.masks.xy:
            corners = extract_corners(polygon, green.shape)
            if corners is None:
                continue

            start_time = time.perf_counter()
            centroid = np.mean(corners, axis=0)  # (x, y) in pixels
            vpi_rectified = rectify_target(vpi_frame, corners)
            num_img = retrieve_numbers(vpi_rectified)
            mod = "Tem"

            conf = 0
            if conf < 0.85:
              with num_img.rlock_cuda() as cuda_buffer:

                tensor = torch.as_tensor(cuda_buffer).unsqueeze(2)
                tensor = tensor.permute(2, 0, 1).float() / 255.0
                # Add batch dimension -> NCHW (1, C, H, W)
                num_tensor = tensor.unsqueeze(0)

                with torch.inference_mode():
                    cnn_output = cnn(num_tensor)
                    probs = torch.softmax(cnn_output, dim=1)
                    print(f"CNN Inference time: {(time.perf_counter() - start_time)*1000:.4f} ms")
                    cnn_conf, cnn_num = torch.max(probs, dim=1)
                    if cnn_conf.item() > conf:
                        mod = "CNN"
                        num = int(cnn_num.item())
                        conf = float(cnn_conf.item())

            if conf < 0.7:
                print("Cannot reliably detect image; skipping")
                continue

            print(f"Number: {num}\nPixel coordinates: {centroid}\nModel: {mod}\nConfidence: {conf:.4f}\n\n")

In [45]:
img = cv.imread("/content/test_hard.jpg")
results = model(img, conf=0.74)
return_points(results, img[:, :, 1])


0: 640x480 6 ps, 20.1ms
Speed: 4.9ms preprocess, 20.1ms inference, 2.3ms postprocess per image at shape (1, 3, 640, 480)
CNN Inference time: 9.8059 ms
Number: 5
Pixel coordinates: [     2277.4        3035]
Model: CNN
Confidence: 0.9998


CNN Inference time: 6.1218 ms
Number: 1
Pixel coordinates: [      648.9      2239.6]
Model: CNN
Confidence: 0.9942


CNN Inference time: 6.7062 ms
Number: 2
Pixel coordinates: [      793.8      1352.9]
Model: CNN
Confidence: 0.7250


CNN Inference time: 6.7283 ms
Number: 4
Pixel coordinates: [     332.32      1403.3]
Model: CNN
Confidence: 0.9976


CNN Inference time: 6.8498 ms
Number: 6
Pixel coordinates: [     2236.5      2211.3]
Model: CNN
Confidence: 0.9990


CNN Inference time: 4.4588 ms
Number: 3
Pixel coordinates: [     1883.7      1126.1]
Model: CNN
Confidence: 0.9870


